# Learned Closure: Training a Neural Viscosity Model Through a PDE Solver

This demo trains a neural network closure **end-to-end through a PDE solver**, with gradients flowing through both the solver and the network during training.

**The setup**: a 1D Burgers' equation solver where the viscosity model — normally a hand-tuned constant — is replaced by a small neural network. The closure is called at every timestep to predict the current viscosity field from the current flow state, and we train the network by differentiating through the entire time-stepping loop.

**The key result**: the learned closure recovers the true (unknown) viscosity profile from solution data alone, and produces better predictions than either the pure physics model (wrong constant viscosity) or a pure ML model (no physics structure).

## Architecture: two Tesseracts, composed in an outer loop

The demo uses two independent Tesseracts:

- **`burgers_solver`**: a single-timestep Burgers' equation solver. Takes the current velocity field `u` and a viscosity field `nu`, returns the velocity field after one explicit Euler step. This is a pure physics component with a clean interface: `(u, nu, dt) → u_next`.
- **`neural_viscosity`**: a small MLP that maps local flow features $(u, \partial u/\partial x, x)$ to a viscosity field $\nu$. Exposes standard gradient endpoints (VJP, JVP, Jacobian).

The outer time-stepping loop lives in the training script and calls both Tesseracts via `apply_tesseract` at every timestep:

```
for each timestep:
    nu_field = apply_tesseract(closure, {u, dudx, x, weights})
    u_next   = apply_tesseract(solver,  {u, nu_field, dt})
    u = u_next
```

Because `apply_tesseract` registers each call as a JAX custom primitive, `jax.grad` through the loop automatically dispatches VJP calls back through both Tesseracts. Gradients flow end-to-end with no manual plumbing.

### Why this architecture matters

The solver's interface — `(u, nu_field, dt) → u_next` — is **not specific to JAX**. A Fortran solver with a hand-written adjoint, or one differentiated by [Enzyme](https://enzyme.mit.edu/) at the LLVM IR level (see the `enzyme_thermal_2d` demo), could implement the same contract. The outer loop and training code would be identical. This is the core Tesseract value proposition: **the closure researcher doesn't need to know how the solver computes its gradients**.

In [ ]:
import sys

sys.path.insert(0, "neural_viscosity")
sys.path.insert(0, "burgers_solver")

import burgers_solver.tesseract_api as solver_api
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import neural_viscosity.tesseract_api as closure_api
import optax
from tesseract_jax import apply_tesseract

from tesseract_core import Tesseract

jax.config.update("jax_enable_x64", True)

# Grid setup (must match the solver)
N = 128
DX = 1.0 / (N - 1)
X = jnp.linspace(0.0, 1.0, N)

# Load both Tesseracts (local dev mode — no Docker needed)
closure_tess = Tesseract.from_tesseract_api("neural_viscosity/tesseract_api.py")
solver_tess = Tesseract.from_tesseract_api("burgers_solver/tesseract_api.py")

print(f"Closure endpoints: {closure_tess.available_endpoints}")
print(f"Solver endpoints:  {solver_tess.available_endpoints}")

## 1. The ground truth: Burgers' equation with spatially-varying viscosity

We generate training data from a Burgers' equation with a **known but non-trivial** viscosity profile:

$$\nu_{\text{true}}(x) = \nu_0 \left(1 + A \sin(\pi x)\right)$$

This represents a spatially-varying material property — analogous to a turbulence closure, constitutive law, or sub-grid model that varies across the domain. The neural closure's job is to recover this profile from solution data alone.

In [ ]:
# True viscosity profile
NU_0 = 0.02
A = 0.8
nu_true = NU_0 * (1.0 + A * jnp.sin(jnp.pi * X))


# Reference solver: Burgers' equation with known viscosity (no neural network).
# Uses the same single-step stencil as the solver Tesseract.
def burgers_reference(u0, nu_field, dt, n_steps):
    """Solve Burgers' equation with a prescribed viscosity field."""

    def step(u, _):
        out = solver_api.apply_jit({"u": u, "nu": nu_field, "dt": dt})
        return out["u_next"], u

    u_final, u_history = jax.lax.scan(step, u0, None, length=n_steps)
    return u_final, u_history


# Generate training data: multiple initial conditions
DT = 5e-5
N_STEPS = 200
key = jax.random.PRNGKey(0)


def make_ic(key):
    """Random smooth initial condition: sum of low-frequency sinusoids."""
    k1, k2, k3 = jax.random.split(key, 3)
    a1 = 0.5 + 0.5 * jax.random.uniform(k1)
    a2 = 0.3 * jax.random.uniform(k2)
    phase = jax.random.uniform(k3) * jnp.pi
    u0 = a1 * jnp.sin(2 * jnp.pi * X + phase) + a2 * jnp.sin(4 * jnp.pi * X)
    # Zero boundary conditions
    u0 = u0.at[0].set(0.0).at[-1].set(0.0)
    return u0


N_TRAIN = 8
N_TEST = 4
keys = jax.random.split(key, N_TRAIN + N_TEST)

train_ics = jnp.stack([make_ic(keys[i]) for i in range(N_TRAIN)])
test_ics = jnp.stack([make_ic(keys[N_TRAIN + i]) for i in range(N_TEST)])

# Generate ground-truth solutions
train_targets = jnp.stack(
    [burgers_reference(ic, nu_true, DT, N_STEPS)[0] for ic in train_ics]
)
test_targets = jnp.stack(
    [burgers_reference(ic, nu_true, DT, N_STEPS)[0] for ic in test_ics]
)

print(f"Training set: {N_TRAIN} initial conditions -> solutions")
print(f"Test set:     {N_TEST} initial conditions -> solutions")
print(f"Grid: {N} points, dt={DT}, {N_STEPS} steps")

# Visualize one example
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(X, train_ics[0], label="Initial condition")
axes[0].plot(X, train_targets[0], label=f"Solution at t={DT * N_STEPS:.3f}")
axes[0].set_xlabel("x")
axes[0].set_ylabel("u")
axes[0].legend()
axes[0].set_title("Example: Burgers' equation with true viscosity")

axes[1].plot(X, nu_true, "k-", linewidth=2)
axes[1].set_xlabel("x")
axes[1].set_ylabel(r"$\nu(x)$")
axes[1].set_title("True viscosity profile (to be learned)")
axes[1].set_ylim(bottom=0)

plt.tight_layout()
plt.show()

## 2. Initialize the neural closure

The closure is a small MLP with 2 hidden layers of 32 units each. Its weights are passed as explicit inputs to the solver so that `jax.grad` can differentiate through the entire pipeline.

In [ ]:
def init_params(key):
    """Initialize neural closure weights with Xavier initialization."""
    keys = jax.random.split(key, 3)
    return {
        "w1": jax.random.normal(keys[0], (3, 32)) * jnp.sqrt(2.0 / 3),
        "b1": jnp.zeros(32),
        "w2": jax.random.normal(keys[1], (32, 32)) * jnp.sqrt(2.0 / 32),
        "b2": jnp.zeros(32),
        "w3": jax.random.normal(keys[2], (32, 1)) * jnp.sqrt(2.0 / 32),
        "b3": jnp.zeros(1),
    }


params = init_params(jax.random.PRNGKey(1))
print(f"Total parameters: {sum(p.size for p in jax.tree.leaves(params))}")

# Verify the closure produces sensible output
test_nu = closure_api.apply_jit(
    {"u": train_ics[0], "dudx": jnp.gradient(train_ics[0], DX), "x": X, **params}
)["nu"]
print(
    f"Initial viscosity range: [{float(test_nu.min()):.4f}, {float(test_nu.max()):.4f}]"
)

## 3. End-to-end training: differentiating through the solver

The training loss is:

$$\mathcal{L}(\theta) = \frac{1}{M} \sum_{i=1}^{M} \left\| u_{\text{solver}}(u_0^{(i)}; \nu_\theta) - u_{\text{data}}^{(i)} \right\|^2$$

where $\nu_\theta$ is the neural viscosity closure with parameters $\theta$, and $u_{\text{solver}}$ runs the full Burgers' equation with $\nu_\theta$ called at every timestep.

The outer loop calls both Tesseracts via `apply_tesseract`:

```python
for each timestep:
    nu = apply_tesseract(closure_tess, {u, dudx, x, weights})["nu"]
    u  = apply_tesseract(solver_tess,  {u, nu, dt})["u_next"]
```

`jax.grad` differentiates through the entire loop — through every timestep, through both `apply_tesseract` calls — automatically. Each `apply_tesseract` dispatches VJP calls back to the respective Tesseract during the backward pass.

In [ ]:
def solve_with_closure(u0, params, dt, n_steps):
    """Run the full time-stepping loop, calling both Tesseracts at each step.

    This is the composition pattern:
      closure: (u, dudx, x, weights) -> nu_field
      solver:  (u, nu_field, dt)     -> u_next

    The solver Tesseract is a pure physics component. In production, it could be
    a Fortran solver with an adjoint — the interface is the same.
    """
    u = u0
    for _step in range(n_steps):
        dudx = jnp.zeros_like(u)
        dudx = dudx.at[1:-1].set((u[2:] - u[:-2]) / (2 * DX))

        # Closure: predict viscosity from current flow state
        closure_out = apply_tesseract(
            closure_tess, {"u": u, "dudx": dudx, "x": X, **params}
        )
        nu = closure_out["nu"]

        # Solver: one explicit Euler step
        solver_out = apply_tesseract(solver_tess, {"u": u, "nu": nu, "dt": dt})
        u = solver_out["u_next"]
    return u


def loss_single(params, u0, target):
    """Loss for a single initial condition: MSE between solver output and data."""
    u_final = solve_with_closure(u0, params, DT, N_STEPS)
    return jnp.mean((u_final - target) ** 2)


def loss_batch(params, ics, targets):
    """Mean loss over a batch of initial conditions."""
    losses = jax.vmap(lambda u0, tgt: loss_single(params, u0, tgt))(ics, targets)
    return jnp.mean(losses)


grad_fn = jax.grad(loss_batch)

# Verify gradients work
print("Testing forward pass + gradient...")
l0 = loss_batch(params, train_ics, train_targets)
g0 = grad_fn(params, train_ics, train_targets)
print(f"  Initial loss: {float(l0):.6e}")
print(
    f"  Gradient norm: {float(jnp.sqrt(sum(jnp.sum(g**2) for g in jax.tree.leaves(g0)))):.6e}"
)
print("  Gradients flow: loss -> solver VJP -> closure VJP -> network weights.")

### Gradient validation against finite differences

Correctness proof: the AD gradients match finite differences to high precision.

In [ ]:
# Finite difference check on a few weight elements
eps = 1e-5
print(f"{'Parameter':>10s} {'Index':>8s} {'AD':>14s} {'FD':>14s} {'Rel. Error':>12s}")
for pname in ["w1", "w2", "w3"]:
    idx = (0, 0)

    def fd_loss(val, _pname=pname, _idx=idx):
        p = {**params, _pname: params[_pname].at[_idx].set(val)}
        return loss_batch(p, train_ics[:2], train_targets[:2])

    v0 = params[pname][idx]
    fd = (fd_loss(v0 + eps) - fd_loss(v0 - eps)) / (2 * eps)
    ad = jax.grad(loss_batch)({**params}, train_ics[:2], train_targets[:2])[pname][idx]
    rel_err = abs(float(ad) - float(fd)) / (abs(float(fd)) + 1e-30)
    print(
        f"{pname:>10s} {idx!s:>8s} {float(ad):14.6e} {float(fd):14.6e} {rel_err:12.2e}"
    )

### Training loop

In [ ]:
N_EPOCHS = 500
LR = 3e-3

# Re-initialize for a clean training run
params = init_params(jax.random.PRNGKey(1))
optimizer = optax.adam(LR)
opt_state = optimizer.init(params)

train_losses = []
test_losses = []


@jax.jit
def train_step(params, opt_state, ics, targets):
    loss, grads = jax.value_and_grad(loss_batch)(params, ics, targets)
    updates, opt_state_new = optimizer.update(grads, opt_state, params)
    params_new = optax.apply_updates(params, updates)
    return params_new, opt_state_new, loss


print("Training neural closure through the solver...")
for epoch in range(N_EPOCHS):
    params, opt_state, train_loss = train_step(
        params, opt_state, train_ics, train_targets
    )
    train_losses.append(float(train_loss))

    if epoch % 50 == 0 or epoch == N_EPOCHS - 1:
        test_loss = float(loss_batch(params, test_ics, test_targets))
        test_losses.append((epoch, test_loss))
        print(
            f"  Epoch {epoch:4d}: train loss = {train_losses[-1]:.4e}, test loss = {test_loss:.4e}"
        )

print(f"\nFinal train loss: {train_losses[-1]:.4e}")
print(f"Final test loss:  {test_losses[-1][1]:.4e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogy(train_losses, label="Train")
test_epochs, test_vals = zip(*test_losses, strict=False)
axes[0].semilogy(test_epochs, test_vals, "o-", label="Test")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Training loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Compare learned viscosity to true viscosity.
# Evaluate the closure Tesseract at several representative flow states.
nu_samples = []
for ic in train_ics:
    dudx = jnp.zeros_like(ic)
    dudx = dudx.at[1:-1].set((ic[2:] - ic[:-2]) / (2 * DX))
    nu_i = apply_tesseract(closure_tess, {"u": ic, "dudx": dudx, "x": X, **params})[
        "nu"
    ]
    nu_samples.append(nu_i)
nu_samples = jnp.stack(nu_samples)
nu_mean = jnp.mean(nu_samples, axis=0)
nu_std = jnp.std(nu_samples, axis=0)

axes[1].plot(X, nu_true, "k-", linewidth=2, label="True viscosity")
axes[1].plot(X, nu_mean, "r--", linewidth=2, label="Learned (mean over ICs)")
axes[1].fill_between(
    X, nu_mean - nu_std, nu_mean + nu_std, color="r", alpha=0.15, label="Learned (std)"
)
axes[1].set_xlabel("x")
axes[1].set_ylabel(r"$\nu$")
axes[1].set_title("Recovered viscosity profile")
axes[1].legend(fontsize=9)
axes[1].set_ylim(bottom=0)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Baselines: why the hybrid model wins

We compare three approaches:

| Model | Description |
|---|---|
| **Constant viscosity** | Burgers' solver with $\nu = \nu_0$ (the standard "wrong" closure) |
| **Direct ML** | MLP trained to map $u_0 \to u_\text{final}$ directly, no physics |
| **Learned closure** | Neural $\nu(u, \partial u/\partial x, x)$ trained through the solver (this demo) |

In [ ]:
# --- Baseline 1: Constant viscosity ---
nu_const = NU_0 * jnp.ones(N)  # Wrong: uniform instead of spatially varying
const_preds_test = jnp.stack(
    [burgers_reference(ic, nu_const, DT, N_STEPS)[0] for ic in test_ics]
)
const_mse = float(jnp.mean((const_preds_test - test_targets) ** 2))
print(f"Constant viscosity test MSE: {const_mse:.4e}")


# --- Baseline 2: Direct ML (MLP mapping u0 -> u_final, no physics) ---
def init_direct_params(key):
    """Larger MLP for direct prediction (more capacity since no physics inductive bias)."""
    keys = jax.random.split(key, 4)
    return {
        "w1": jax.random.normal(keys[0], (N, 128)) * jnp.sqrt(2.0 / N),
        "b1": jnp.zeros(128),
        "w2": jax.random.normal(keys[1], (128, 128)) * jnp.sqrt(2.0 / 128),
        "b2": jnp.zeros(128),
        "w3": jax.random.normal(keys[2], (128, 64)) * jnp.sqrt(2.0 / 128),
        "b3": jnp.zeros(64),
        "w4": jax.random.normal(keys[3], (64, N)) * jnp.sqrt(2.0 / 64),
        "b4": jnp.zeros(N),
    }


def direct_predict(params, u0):
    """Pure ML: MLP maps u0 directly to u_final."""
    h = jnp.tanh(u0 @ params["w1"] + params["b1"])
    h = jnp.tanh(h @ params["w2"] + params["b2"])
    h = jnp.tanh(h @ params["w3"] + params["b3"])
    return h @ params["w4"] + params["b4"]


def direct_loss(params, ics, targets):
    preds = jax.vmap(lambda u0: direct_predict(params, u0))(ics)
    return jnp.mean((preds - targets) ** 2)


# Train the direct ML baseline
direct_params = init_direct_params(jax.random.PRNGKey(2))
direct_opt = optax.adam(1e-3)
direct_opt_state = direct_opt.init(direct_params)


@jax.jit
def direct_train_step(params, opt_state, ics, targets):
    loss, grads = jax.value_and_grad(direct_loss)(params, ics, targets)
    updates, opt_state_new = direct_opt.update(grads, opt_state, params)
    params_new = optax.apply_updates(params, updates)
    return params_new, opt_state_new, loss


print("\nTraining direct ML baseline...")
direct_losses = []
for epoch in range(500):
    direct_params, direct_opt_state, dl = direct_train_step(
        direct_params, direct_opt_state, train_ics, train_targets
    )
    direct_losses.append(float(dl))
    if epoch % 100 == 0:
        test_dl = float(direct_loss(direct_params, test_ics, test_targets))
        print(
            f"  Epoch {epoch:4d}: train = {direct_losses[-1]:.4e}, test = {test_dl:.4e}"
        )

direct_test_mse = float(direct_loss(direct_params, test_ics, test_targets))
print(f"\nDirect ML test MSE: {direct_test_mse:.4e}")

# --- Learned closure (already trained above) ---
learned_test_mse = float(loss_batch(params, test_ics, test_targets))
print(f"Learned closure test MSE: {learned_test_mse:.4e}")

print(f"\n{'Model':<25s} {'Test MSE':>12s}")
print("-" * 40)
print(f"{'Constant viscosity':<25s} {const_mse:12.4e}")
print(f"{'Direct ML':<25s} {direct_test_mse:12.4e}")
print(f"{'Learned closure':<25s} {learned_test_mse:12.4e}")

## 5. Solution comparison on test data

The key visual: for an unseen initial condition, compare the three models' predictions against the ground truth.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for idx in range(4):
    ax = axes[idx // 2, idx % 2]
    ic = test_ics[idx]
    target = test_targets[idx]

    # Constant viscosity prediction
    const_pred = burgers_reference(ic, nu_const, DT, N_STEPS)[0]

    # Direct ML prediction
    direct_pred = direct_predict(direct_params, ic)

    # Learned closure prediction (outer loop calling both Tesseracts)
    learned_pred = solve_with_closure(ic, params, DT, N_STEPS)

    ax.plot(X, target, "k-", linewidth=2, label="Ground truth")
    ax.plot(X, const_pred, "b--", linewidth=1.5, alpha=0.7, label="Constant viscosity")
    ax.plot(X, direct_pred, "g:", linewidth=1.5, alpha=0.7, label="Direct ML")
    ax.plot(X, learned_pred, "r-", linewidth=1.5, label="Learned closure")
    ax.set_xlabel("x")
    ax.set_ylabel("u")
    ax.set_title(f"Test case {idx + 1}")
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=9)

plt.suptitle(
    "Solution predictions on unseen initial conditions", fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

## 6. Modularity: swap the closure *or the solver*

The solver and closure are independent Tesseracts with a clean contract: the solver takes `(u, nu_field, dt)` and returns `u_next`. The closure takes `(u, dudx, x, weights)` and returns `nu`.

This means you can:
- **Swap the closure**: replace the neural network architecture without touching the solver
- **Swap the solver**: replace the JAX solver with a Fortran solver (differentiated by Enzyme or a hand-written adjoint) without touching the closure or training loop

Here we demonstrate closure swapping — training a different closure initialization against the same solver.

In [ ]:
# Swap: re-initialize with a different random seed (different starting point).
# In production with Tesseract containers, this would mean pointing the solver
# at a completely different closure Tesseract URL. The solver code is untouched.

swapped_params = init_params(jax.random.PRNGKey(99))  # different seed
swapped_opt = optax.adam(3e-3)
swapped_opt_state = swapped_opt.init(swapped_params)

print("Training a different closure (same solver, different initialization)...")
for _epoch in range(500):
    swapped_params, swapped_opt_state, sl = train_step(
        swapped_params, swapped_opt_state, train_ics, train_targets
    )

swapped_test_mse = float(loss_batch(swapped_params, test_ics, test_targets))
print(f"  Swapped closure test MSE:  {swapped_test_mse:.4e}")
print(f"  Original closure test MSE: {learned_test_mse:.4e}")
print("\nSolver code: unchanged. Only the closure was swapped.")

# Compare the two learned viscosity profiles
nu_orig = apply_tesseract(
    closure_tess,
    {"u": train_ics[0], "dudx": jnp.gradient(train_ics[0], DX), "x": X, **params},
)["nu"]
nu_swap = apply_tesseract(
    closure_tess,
    {
        "u": train_ics[0],
        "dudx": jnp.gradient(train_ics[0], DX),
        "x": X,
        **swapped_params,
    },
)["nu"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(X, nu_true, "k-", linewidth=2, label="True")
ax.plot(X, nu_orig, "r--", linewidth=1.5, label="Closure A")
ax.plot(X, nu_swap, "b:", linewidth=1.5, label="Closure B (swapped)")
ax.set_xlabel("x")
ax.set_ylabel(r"$\nu$")
ax.set_title("Two different closures, same solver")
ax.legend()
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| What | How |
|---|---|
| Two independent Tesseracts | Solver (single-timestep PDE) + closure (neural viscosity) |
| Composed in an outer loop | `apply_tesseract(closure, ...)` then `apply_tesseract(solver, ...)` at each timestep |
| End-to-end gradients | `jax.grad` dispatches VJP through both Tesseracts automatically |
| Gradient correctness | Validated against finite differences |
| Learned closure beats baselines | Lower test MSE than constant viscosity or direct ML |
| Modular swapping | Change either Tesseract without touching the other |

### Line of sight to production

The solver Tesseract's interface — `(u, nu_field, dt) → u_next` with VJP — is not JAX-specific. A Fortran solver differentiated by [Enzyme](https://enzyme.mit.edu/) (see the `enzyme_thermal_2d` demo) or with a hand-written discrete adjoint could implement the same contract. The training loop and closure Tesseract would be **identical**. This is the core value proposition: closure researchers get access to a library of differentiable solvers without learning each solver's internals.

### What's next

- **Containerized deployment**: Build each Tesseract as a Docker image. The outer loop calls both over HTTP via `apply_tesseract` — same code, real container isolation.
- **Legacy solver integration**: Wrap a Fortran/C++ solver with adjoint as a Tesseract. The closure training loop above works unchanged.
- **Scale up**: Larger grids, 2D/3D problems, more complex closures (e.g., convolutional, attention-based).
- **Real applications**: Replace the Burgers' equation with a turbulence model, climate sub-grid scheme, or materials constitutive law.